# Delta Lake Setup

This notebook sets up the initial Delta Lake table using the Global Superstore dataset. We'll:

1. Load the Global Superstore dataset from CSV
2. Define an appropriate schema with data types for all columns
3. Partition the data by "Ship Mode" and "Category" for performance
4. Write the data to a Delta table with optimization settings
5. Create Z-ordering on frequently queried columns
6. Set up table properties for retention and other Delta features
7. Print table metadata and sample data for verification
8. Include statistics on the initial data load

## 1. Initialize Spark Session with Delta Lake

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set SPARK_HOME environment variable
os.environ['SPARK_HOME'] = '/usr/local/lib/python3.10/site-packages/pyspark'

# Import PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Create Spark session with Delta Lake support
spark = SparkSession.builder \
    .appName("Delta Lake Setup") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
    .config("spark.databricks.delta.autoCompact.enabled", "true") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
try:
    delta_version = spark.sql("SELECT version() as delta_version").collect()[0][0]
    print(f"Delta Lake version: {delta_version}")
except:
    print("Could not determine Delta Lake version")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09a92013-d334-4f24-a9ab-333e401d835f;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: resolution report :: resolve 196ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-09a92013-d334-4f24-a9ab-333e401d835f
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/6ms)
25/04/30 19:49:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/04/30 19:49:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.4.0


Delta Lake version: 3.4.0 87a5442f7ed96b11051d8a9333476d080054e5a0


## 2. Define Constants and Paths

In [2]:
# Define paths
DATA_DIR = "/opt/spark/data"
RAW_DATA_PATH = os.path.join(DATA_DIR, "raw/Global_Superstore.csv")
DELTA_TABLE_PATH = os.path.join(DATA_DIR, "processed/global_superstore_delta")

# Create directories if they don't exist
os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
os.makedirs(os.path.dirname(DELTA_TABLE_PATH), exist_ok=True)

print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Delta table path: {DELTA_TABLE_PATH}")

Raw data path: /opt/spark/data/raw/Global_Superstore.csv
Delta table path: /opt/spark/data/processed/global_superstore_delta


## 3. Generate Sample Data if Needed

In [3]:
# Function to generate sample data
def generate_sample_data(num_records=1000):
    """Generate sample data for the Global Superstore dataset."""
    # Base data generation
    data = {
        'Row_ID': range(1, num_records + 1),
        'Order_ID': [f"ORD-{i:05d}" for i in range(1, num_records + 1)],
        'Order_Date': [(datetime.now() - timedelta(days=np.random.randint(1, 365))).strftime('%Y-%m-%d') for _ in range(num_records)],
        'Ship_Date': [(datetime.now() - timedelta(days=np.random.randint(1, 30))).strftime('%Y-%m-%d') for _ in range(num_records)],
        'Ship_Mode': np.random.choice(['Standard Class', 'First Class', 'Second Class', 'Same Day'], num_records),
        'Customer_ID': [f"CUST-{i:05d}" for i in range(1, num_records + 1)],
        'Customer_Name': [f"Customer {i}" for i in range(1, num_records + 1)],
        'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], num_records),
        'Country': np.random.choice(['United States', 'Canada', 'Mexico', 'United Kingdom', 'France'], num_records),
        'City': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], num_records),
        'State': np.random.choice(['New York', 'California', 'Illinois', 'Texas', 'Arizona'], num_records),
        'Postal_Code': np.random.choice(['10001', '90001', '60601', '77001', '85001'], num_records),
        'Region': np.random.choice(['East', 'West', 'Central', 'South'], num_records),
        'Product_ID': [f"PROD-{i:05d}" for i in range(1, num_records + 1)],
        'Category': np.random.choice(['Furniture', 'Office Supplies', 'Technology'], num_records),
        'Sub_Category': np.random.choice(['Chairs', 'Tables', 'Phones', 'Storage', 'Binders'], num_records),
        'Product_Name': [f"Product {i}" for i in range(1, num_records + 1)],
        'Sales': np.random.uniform(10, 1000, num_records).round(2),
        'Quantity': np.random.randint(1, 10, num_records),
        'Discount': np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], num_records),
        'Profit': np.random.uniform(-100, 500, num_records).round(2)
    }
    return pd.DataFrame(data)

# Check if the raw data file exists
if not os.path.exists(RAW_DATA_PATH):
    print(f"Raw data file not found at {RAW_DATA_PATH}")
    print("Generating sample data...")
    
    # Generate sample data
    sample_data = generate_sample_data(num_records=1000)
    
    # Save to CSV
    sample_data.to_csv(RAW_DATA_PATH, index=False)
    print(f"Generated {len(sample_data)} sample records and saved to {RAW_DATA_PATH}")
else:
    print(f"Raw data file found at {RAW_DATA_PATH}")
    
    # Read the CSV file to check column names
    raw_df = pd.read_csv(RAW_DATA_PATH)
    print(f"Loaded {len(raw_df)} records from {RAW_DATA_PATH}")
    
    # Check if column names contain spaces
    columns_with_spaces = [col for col in raw_df.columns if ' ' in col]
    if columns_with_spaces:
        print(f"Found {len(columns_with_spaces)} columns with spaces: {columns_with_spaces}")
        print("Renaming columns to replace spaces with underscores...")
        
        # Rename columns
        raw_df.columns = [col.replace(' ', '_') for col in raw_df.columns]
        
        # Save the updated CSV
        raw_df.to_csv(RAW_DATA_PATH, index=False)
        print(f"Updated CSV file with renamed columns")
    else:
        print("No columns with spaces found")

Raw data file found at /opt/spark/data/raw/Global_Superstore.csv
Loaded 10000 records from /opt/spark/data/raw/Global_Superstore.csv
No columns with spaces found


## 4. Define Schema for Global Superstore Dataset

In [4]:
# Define schema for the Global Superstore dataset with underscores instead of spaces
schema = StructType([
    StructField("Row_ID", IntegerType(), False),
    StructField("Order_ID", StringType(), False),
    StructField("Order_Date", DateType(), False),
    StructField("Ship_Date", DateType(), False),
    StructField("Ship_Mode", StringType(), False),
    StructField("Customer_ID", StringType(), False),
    StructField("Customer_Name", StringType(), False),
    StructField("Segment", StringType(), False),
    StructField("Country", StringType(), False),
    StructField("City", StringType(), False),
    StructField("State", StringType(), True),
    StructField("Postal_Code", StringType(), True),
    StructField("Region", StringType(), False),
    StructField("Product_ID", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Sub_Category", StringType(), False),
    StructField("Product_Name", StringType(), False),
    StructField("Sales", DoubleType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Discount", DoubleType(), False),
    StructField("Profit", DoubleType(), False)
])

print(f"Schema defined with {len(schema.fields)} fields")

Schema defined with 21 fields


## 5. Load and Prepare Data

In [5]:
# Load data with the defined schema
try:
    # Read CSV with date format
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .schema(schema) \
        .option("dateFormat", "yyyy-MM-dd") \
        .load(RAW_DATA_PATH)
    
    # Convert date strings to date type if needed
    if df.schema["Order_Date"].dataType.typeName() == "string":
        df = df.withColumn("Order_Date", to_date(col("Order_Date"), "yyyy-MM-dd"))
    if df.schema["Ship_Date"].dataType.typeName() == "string":
        df = df.withColumn("Ship_Date", to_date(col("Ship_Date"), "yyyy-MM-dd"))
    
    # Show sample data
    print(f"Loaded {df.count()} records from {RAW_DATA_PATH}")
    df.printSchema()
    df.show(5)
except Exception as e:
    print(f"Error loading data: {e}")
    # Try with inferred schema as fallback
    print("Trying with inferred schema...")
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(RAW_DATA_PATH)
    
    # Convert date strings to date type
    if "Order_Date" in df.columns:
        df = df.withColumn("Order_Date", to_date(col("Order_Date"), "yyyy-MM-dd"))
    elif "Order Date" in df.columns:
        df = df.withColumnRenamed("Order Date", "Order_Date")
        df = df.withColumn("Order_Date", to_date(col("Order_Date"), "yyyy-MM-dd"))
        
    if "Ship_Date" in df.columns:
        df = df.withColumn("Ship_Date", to_date(col("Ship_Date"), "yyyy-MM-dd"))
    elif "Ship Date" in df.columns:
        df = df.withColumnRenamed("Ship Date", "Ship_Date")
        df = df.withColumn("Ship_Date", to_date(col("Ship_Date"), "yyyy-MM-dd"))
    
    # Rename all columns to replace spaces with underscores
    for col_name in df.columns:
        if " " in col_name:
            df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))
    
    print(f"Loaded {df.count()} records with inferred schema")
    df.printSchema()
    df.show(5)

Loaded 10000 records from /opt/spark/data/raw/Global_Superstore.csv
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



25/04/30 19:49:10 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 14, schema size: 21
CSV file: file:///opt/spark/data/raw/Global_Superstore.csv


+------+----------+----------+----------+--------------+-----------+---------------+-----------+---------------+-------+------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|  Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|    Segment|        Country|   City| State|Postal_Code|Region|Product_ID|Category|Sub_Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+----------+----------+----------+--------------+-----------+---------------+-----------+---------------+-------+------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     1|TEST-34607|2023-09-12|2023-07-23|      Same Day|   CG-14825|Test-Customer-0|  Corporate|      Furniture|Storage|461.54|          7|   0.4|    348.98|    null|        null|        null| null|    null|    null|  null|
|     2|TEST-98890|2023-11-17|2023-12-15|  Second Class|   CG-18292|Test-Customer-1|Home Office|     Tec

## 6. Write to Delta Table with Partitioning

In [6]:
# Add metadata columns
df = df.withColumn("Created_At", lit(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")))
df = df.withColumn("Updated_At", lit(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")))
df = df.withColumn("Source", lit("initial_load"))

# Write to Delta table with partitioning
try:
    print(f"Writing data to Delta table at {DELTA_TABLE_PATH}")
    df.write \
        .format("delta") \
        .partitionBy("Ship_Mode", "Category") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    
    print("Data written successfully to Delta table")
except Exception as e:
    print(f"Error writing to Delta table: {e}")
    # Try without partitioning as fallback
    print("Trying without partitioning...")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    print("Data written successfully to Delta table without partitioning")

Writing data to Delta table at /opt/spark/data/processed/global_superstore_delta


25/04/30 19:49:11 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 14, schema size: 21
CSV file: file:///opt/spark/data/raw/Global_Superstore.csv


25/04/30 19:49:18 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Data written successfully to Delta table


## 7. Optimize and Z-Order the Delta Table

In [7]:
# Optimize and Z-Order the Delta table
try:
    print("Optimizing Delta table...")
    spark.sql(f"OPTIMIZE delta.`{DELTA_TABLE_PATH}`").show()
    
    print("Z-Ordering Delta table by Order_Date, Customer_ID, and Product_ID...")
    spark.sql(f"OPTIMIZE delta.`{DELTA_TABLE_PATH}` ZORDER BY (Order_Date, Customer_ID, Product_ID)").show()
    
    print("Optimization and Z-Ordering completed successfully")
except Exception as e:
    print(f"Error optimizing Delta table: {e}")

Optimizing Delta table...


+--------------------+--------------------+
|                path|             metrics|
+--------------------+--------------------+
|file:/opt/spark/d...|{0, 0, {null, nul...|
+--------------------+--------------------+

Z-Ordering Delta table by Order_Date, Customer_ID, and Product_ID...


Error optimizing Delta table: An error occurred while calling o40.sql.
: io.delta.exceptions.ConcurrentDeleteReadException: This transaction attempted to read one or more files that were deleted (for example Ship_Mode=Same%20Day/Category=__HIVE_DEFAULT_PARTITION__/part-00000-f6a105c3-4b16-4c3d-87a1-cf147b4d7cf4.c000.snappy.parquet in partition [Ship_Mode=Same Day, Category=null]) by a concurrent update. Please try the operation again.
Conflicting commit: {"timestamp":1746042566183,"operation":"WRITE","operationParameters":{"mode":Overwrite,"partitionBy":[]},"readVersion":0,"isolationLevel":"Serializable","isBlindAppend":false,"operationMetrics":{"numFiles":"4","numOutputRows":"10000","numOutputBytes":"398507"},"engineInfo":"Apache-Spark/3.4.0 Delta-Lake/2.4.0","txnId":"5d0b38c9-0fdc-465d-9d3b-d6ae0b60cc5c"}
Refer to https://docs.delta.io/latest/concurrency-control.html for more details.
	at org.apache.spark.sql.delta.DeltaErrorsBase.concurrentDeleteReadException(DeltaErrors.scala:2195)

25/04/30 19:49:31 WARN OptimizeExecutor: The following compacted files were delete during checkpoint Ship_Mode=Same%20Day/Category=__HIVE_DEFAULT_PARTITION__/part-00000-f6a105c3-4b16-4c3d-87a1-cf147b4d7cf4.c000.snappy.parquet,Ship_Mode=First%20Class/Category=__HIVE_DEFAULT_PARTITION__/part-00000-3dcfd0c6-a97c-4be3-815c-e0fd107b561b.c000.snappy.parquet,Ship_Mode=Second%20Class/Category=__HIVE_DEFAULT_PARTITION__/part-00000-5662efc0-bce1-4253-809f-d9e23f15c875.c000.snappy.parquet,Ship_Mode=Standard%20Class/Category=__HIVE_DEFAULT_PARTITION__/part-00000-07e763d6-2736-4599-bb1f-7675d5a8cb41.c000.snappy.parquet. Aborting the compaction.
25/04/30 19:49:31 WARN OptimizeExecutor: Semantic conflicts detected. Aborting operation.


## 8. Set Delta Table Properties

In [8]:
# Set Delta table properties
try:
    print("Setting Delta table properties...")
    
    # Set retention period to 30 days
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.logRetentionDuration' = '30 days')").show()
    
    # Enable auto compaction
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')").show()
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true')").show()
    
    # Set description
    spark.sql(f"ALTER TABLE delta.`{DELTA_TABLE_PATH}` SET TBLPROPERTIES ('comment' = 'Global Superstore dataset for Delta Lake demo')").show()
    
    print("Delta table properties set successfully")
except Exception as e:
    print(f"Error setting Delta table properties: {e}")

Setting Delta table properties...


++
||
++
++



Error setting Delta table properties: Unknown configuration was specified: delta.autoOptimize.optimizeWrite
To disable this check, set spark.databricks.delta.allowArbitraryProperties.enabled=true in the Spark session configuration.


## 9. Verify Delta Table

In [9]:
# Verify Delta table
try:
    print("Verifying Delta table...")
    
    # Get table details
    print("\nTable Details:")
    spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_TABLE_PATH}`").show(truncate=False)
    
    # Get table history
    print("\nTable History:")
    spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_TABLE_PATH}`").show(truncate=False)
    
    # Get record count
    count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
    print(f"\nTotal records: {count}")
    
    # Show sample data
    print("\nSample Data:")
    spark.read.format("delta").load(DELTA_TABLE_PATH).show(5)
    
    print("Delta table verification completed successfully")
except Exception as e:
    print(f"Error verifying Delta table: {e}")

Verifying Delta table...

Table Details:


+------+------------------------------------+----+-----------+------------------------------------------------------+-----------------------+-----------------------+---------------------+--------+-----------+---------------------------------------+----------------+----------------+------------------------+
|format|id                                  |name|description|location                                              |createdAt              |lastModified           |partitionColumns     |numFiles|sizeInBytes|properties                             |minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+----+-----------+------------------------------------------------------+-----------------------+-----------------------+---------------------+--------+-----------+---------------------------------------+----------------+----------------+------------------------+
|delta |6f807244-5d42-4933-a276-740fd82c6bc5|null|null       |file:/opt/spar

+-------+-----------------------+------+--------+-----------------+------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation        |operationParameters                                         |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                 |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+-----------------+------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------+------------+-----------------------------------+
|2      |2025-04-30 19:49:32.207|null  |null    |SET TBLPROPERTIES|{prope


Total records: 10000

Sample Data:


+------+----------+----------+----------+--------------+-----------+----------------+-----------+---------------+-------+------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------------------+-------------------+------------+
|Row_ID|  Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|    Segment|        Country|   City| State|Postal_Code|Region|Product_ID|Category|Sub_Category|Product_Name|Sales|Quantity|Discount|Profit|         Created_At|         Updated_At|      Source|
+------+----------+----------+----------+--------------+-----------+----------------+-----------+---------------+-------+------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------------------+-------------------+------------+
|     3|TEST-73091|2023-07-14|2023-02-28|Standard Class|   CG-50530| Test-Customer-2|   Consumer|      Furniture| Chairs|377.72|          1|   0.3|     96.01|    null

## 10. Generate Statistics

In [10]:
# Generate statistics
try:
    print("Generating statistics...")
    
    # Read Delta table
    delta_df = spark.read.format("delta").load(DELTA_TABLE_PATH)
    
    # Count by Ship Mode
    print("\nCount by Ship Mode:")
    delta_df.groupBy("Ship_Mode").count().orderBy("count", ascending=False).show()
    
    # Count by Category
    print("\nCount by Category:")
    delta_df.groupBy("Category").count().orderBy("count", ascending=False).show()
    
    # Sales statistics
    print("\nSales Statistics:")
    delta_df.select("Sales").summary().show()
    
    # Profit statistics
    print("\nProfit Statistics:")
    delta_df.select("Profit").summary().show()
    
    print("Statistics generation completed successfully")
except Exception as e:
    print(f"Error generating statistics: {e}")

Generating statistics...



Count by Ship Mode:


+--------------+-----+
|     Ship_Mode|count|
+--------------+-----+
|      Same Day| 2520|
|Standard Class| 2513|
|  Second Class| 2495|
|   First Class| 2472|
+--------------+-----+


Count by Category:


+--------+-----+
|Category|count|
+--------+-----+
|    null|10000|
+--------+-----+


Sales Statistics:


+-------+-----+
|summary|Sales|
+-------+-----+
|  count|    0|
|   mean| null|
| stddev| null|
|    min| null|
|    25%| null|
|    50%| null|
|    75%| null|
|    max| null|
+-------+-----+


Profit Statistics:


+-------+------+
|summary|Profit|
+-------+------+
|  count|     0|
|   mean|  null|
| stddev|  null|
|    min|  null|
|    25%|  null|
|    50%|  null|
|    75%|  null|
|    max|  null|
+-------+------+

Statistics generation completed successfully


## 11. Setup Complete

In [11]:
print("Delta Lake setup completed successfully!")
print(f"Delta table is available at: {DELTA_TABLE_PATH}")
print("You can now proceed with the other notebooks in the demo.")

Delta Lake setup completed successfully!
Delta table is available at: /opt/spark/data/processed/global_superstore_delta
You can now proceed with the other notebooks in the demo.
